In [ ]:
import os
import shutil
import zipfile

# ============================================================ #
# Eval config giống train.ipynb: chỉ giữ path/hyperparameter cần cho eval
# IL_DER dùng 1 global model/checkpoint, không cần chia test theo clients.
# ============================================================ #
CONFIG = {
    "data_dir": "/kaggle/input/datasets/khoilv2005/100-clients/100-clients",
    # Nếu bạn upload zip merge vào Kaggle input, sửa path này.
    "checkpoint_zip": "/kaggle/input/datasets/khoilv2005/il-der/results_il_der_0_5_merged.zip",
    # Nếu Kaggle input đã giải nén sẵn, sửa path này.
    "checkpoint_root": "/kaggle/input/datasets/khoilv2005/il-der/results_il_der",
    "extract_root": "/kaggle/working/il_der_checkpoints",
    "output_dir": "/kaggle/working/results_il_der_eval",
    "num_clients": 100,
    "total_classes": 34,
    "base_classes": 6,
    "classes_per_task": 6,
    "eval_batch_size": 8192,
}

DATA_DIR = globals().get("DATA_DIR", CONFIG["data_dir"])
CHECKPOINT_ZIP = globals().get("CHECKPOINT_ZIP", CONFIG["checkpoint_zip"])
CHECKPOINT_ROOT = globals().get("CHECKPOINT_ROOT", CONFIG["checkpoint_root"])
EXTRACT_ROOT = globals().get("EXTRACT_ROOT", CONFIG["extract_root"])
EVAL_OUTPUT_DIR = globals().get("EVAL_OUTPUT_DIR", CONFIG["output_dir"])

# ============================================================ #
# Eval options - sửa ở đây
# ============================================================ #
# Chọn task train/checkpoint để eval. Mặc định chỉ task 5 để bổ sung IL_DER f1_weighted còn thiếu.
EVAL_TASKS = globals().get("EVAL_TASKS", [5])
# Chọn round train/checkpoint để eval. None = all rounds. Ví dụ: 19, [0, 9, 19], "0,9,19".
EVAL_ROUNDS = globals().get("EVAL_ROUNDS", None)
# IL_DER không dùng per-client split; giữ option để cùng cấu trúc với eval_5.
EVAL_CLIENT_FRACTION = float(globals().get("EVAL_CLIENT_FRACTION", 1.0))
EVAL_CLIENT_SELECTION = globals().get("EVAL_CLIENT_SELECTION", "deterministic")
EVAL_CLIENT_SEED = int(globals().get("EVAL_CLIENT_SEED", 42))
# IL_DER không cần nhiều seed; nhiều seed chỉ chạy lặp cùng kết quả.
EVAL_SPLIT_SEEDS = globals().get("EVAL_SPLIT_SEEDS", [42])
EVAL_PROGRESS_EVERY_CLIENTS = int(globals().get("EVAL_PROGRESS_EVERY_CLIENTS", 0))
EVAL_PROGRESS_EVERY_BATCHES = int(globals().get("EVAL_PROGRESS_EVERY_BATCHES", 0))

# Nếu có zip merge thì giải nén và tự tìm folder chứa checkpoint.
if CHECKPOINT_ZIP and os.path.exists(CHECKPOINT_ZIP):
    if os.path.exists(EXTRACT_ROOT):
        shutil.rmtree(EXTRACT_ROOT)
    os.makedirs(EXTRACT_ROOT, exist_ok=True)
    print("Extracting checkpoint zip:", CHECKPOINT_ZIP)
    with zipfile.ZipFile(CHECKPOINT_ZIP, "r") as zf:
        zf.extractall(EXTRACT_ROOT)
    pt_dirs = sorted({root for root, _, files in os.walk(EXTRACT_ROOT) if any(f.endswith(".pt") for f in files)})
    if not pt_dirs:
        raise FileNotFoundError(f"No .pt files found after extracting {CHECKPOINT_ZIP}")
    # Ưu tiên folder results_il_der nếu có.
    preferred = [p for p in pt_dirs if os.path.basename(p) == "results_il_der"]
    CHECKPOINT_ROOT = preferred[0] if preferred else pt_dirs[0]
elif not os.path.exists(CHECKPOINT_ROOT):
    raise FileNotFoundError(
        "No checkpoint source found. Set CHECKPOINT_ZIP to results_il_der_0_5_merged.zip "
        f"or CHECKPOINT_ROOT to extracted results_il_der. Tried zip={CHECKPOINT_ZIP}, root={CHECKPOINT_ROOT}"
    )

os.makedirs(EVAL_OUTPUT_DIR, exist_ok=True)

print("DATA_DIR:", DATA_DIR)
print("DATA_DIR exists:", os.path.exists(DATA_DIR))
print("CHECKPOINT_ZIP:", CHECKPOINT_ZIP)
print("CHECKPOINT_ROOT:", CHECKPOINT_ROOT)
print("CHECKPOINT_ROOT exists:", os.path.exists(CHECKPOINT_ROOT))
print("EVAL_OUTPUT_DIR:", EVAL_OUTPUT_DIR)
print("EVAL_TASKS:", EVAL_TASKS)
print("EVAL_ROUNDS:", EVAL_ROUNDS)
print("EVAL_SPLIT_SEEDS:", EVAL_SPLIT_SEEDS)


In [ ]:
import os

print("CHECKPOINT_ROOT:", CHECKPOINT_ROOT)
print("exists:", os.path.exists(CHECKPOINT_ROOT))

if os.path.exists(CHECKPOINT_ROOT):
    print("\nTop-level contents:")
    for name in sorted(os.listdir(CHECKPOINT_ROOT)):
        path = os.path.join(CHECKPOINT_ROOT, name)
        kind = "DIR " if os.path.isdir(path) else "FILE"
        size = os.path.getsize(path) if os.path.isfile(path) else ""
        print(f"{kind} {name} {size}")

    print("\n.pt/.pth preview under CHECKPOINT_ROOT:")
    count = 0
    for root, _, files in os.walk(CHECKPOINT_ROOT):
        for filename in sorted(files):
            if filename.lower().endswith((".pt", ".pth")):
                print(os.path.join(root, filename))
                count += 1
                if count >= 80:
                    print("... truncated at 80 files")
                    break
        if count >= 80:
            break
    print("pt/pth shown:", count)


In [ ]:
import gc
import json
import os
import re
import shutil
import sys
import time
from collections import OrderedDict

REPO_PATH = "/tmp/FL_IL_IDS"
CHECKPOINT_ROOT = globals().get("CHECKPOINT_ROOT", "/kaggle/input/datasets/khoilv2005/il-der/results_il_der")
DATA_DIR = globals().get("DATA_DIR", "/kaggle/input/datasets/khoilv2005/100-clients/100-clients")
OUTPUT_DIR = globals().get("EVAL_OUTPUT_DIR", os.path.join(CHECKPOINT_ROOT, "eval_outputs"))
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Select tasks to evaluate. Default = task 5 only.
EVAL_TASKS = globals().get("EVAL_TASKS", [5])
# Select rounds to evaluate. None = all rounds. Examples: 19, [0, 9, 19], "0,9,19".
EVAL_ROUNDS = globals().get("EVAL_ROUNDS", None)
# IL_DER không dùng client fraction; giữ để cùng cấu trúc với eval_5.
EVAL_CLIENT_FRACTION = float(globals().get("EVAL_CLIENT_FRACTION", 1.0))
EVAL_CLIENT_SELECTION = globals().get("EVAL_CLIENT_SELECTION", "deterministic")
EVAL_CLIENT_SEED = int(globals().get("EVAL_CLIENT_SEED", 42))
EVAL_SPLIT_SEEDS = globals().get("EVAL_SPLIT_SEEDS", [42])
EVAL_PROGRESS_EVERY_CLIENTS = int(globals().get("EVAL_PROGRESS_EVERY_CLIENTS", 0))
EVAL_PROGRESS_EVERY_BATCHES = int(globals().get("EVAL_PROGRESS_EVERY_BATCHES", 0))

def setup_imports():
    if os.path.exists(REPO_PATH):
        print(f"Removing stale clone at {REPO_PATH}...")
        shutil.rmtree(REPO_PATH)
    print("Cloning from GitHub...")
    os.system(f"git clone https://github.com/khoilv2005/FL_IL_IDS.git {REPO_PATH}")
    kaggle_prefix = "/kaggle/input"
    sys.path = [REPO_PATH] + [p for p in sys.path if not p.startswith(kaggle_prefix)]
    for name in list(sys.modules.keys()):
        if name == "fed_learning" or name.startswith("fed_learning."):
            del sys.modules[name]
    print("sys.path[0]:", sys.path[0])

setup_imports()

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)

from eval_checkpoint import _make_model, _make_denice_client_model
from fed_learning.servers.nice_server import ContextDetector, build_pooled_context_detector
from fed_learning.training.checkpoint_state import restore_context_detector
from fed_learning.data.incremental_loader import IncrementalDataLoader
from fed_learning.training.local_task_loop import (
    _apply_local_nice_context_mask,
    _nice_seen_mask,
)
from fed_learning.training.denice_eval import (
    evaluate_denice_model,
    _denice_routed_logits_with_episodes,
)
from fed_learning.training.denice_delta_checkpoint import load_denice_checkpoint

def resolve_data_dir(data_dir):
    metadata_path = os.path.join(data_dir, "metadata.json")
    test_path = os.path.join(data_dir, "global_test_data.npz")
    if os.path.exists(metadata_path) and os.path.exists(test_path):
        return data_dir

    print("DATA_DIR invalid or incomplete:", data_dir)
    print("  metadata exists:", os.path.exists(metadata_path))
    print("  global_test_data exists:", os.path.exists(test_path))
    print("Searching Drive for folders containing metadata.json + global_test_data.npz...")

    search_roots = [
        globals().get("DRIVE_ROOT", "/content/drive/MyDrive"),
        "/content/drive/MyDrive",
    ]
    candidates = []
    for root in search_roots:
        if not root or not os.path.exists(root):
            continue
        for dirpath, _, files in os.walk(root):
            file_set = set(files)
            if "metadata.json" in file_set and "global_test_data.npz" in file_set:
                candidates.append(dirpath)
    candidates = sorted(set(candidates))
    print("Data dir candidates:")
    for c in candidates[:30]:
        print("  ", c)
    if not candidates:
        raise FileNotFoundError(
            "No folder with metadata.json and global_test_data.npz found. "
            "Set DATA_DIR to the exact federated split folder."
        )

    preferred = [c for c in candidates if "100" in c and "client" in c.lower()]
    selected = preferred[0] if preferred else candidates[0]
    print("Selected DATA_DIR:", selected)
    return selected

def _parse_eval_tasks(value):
    if value is None or value == "":
        return None
    if isinstance(value, int):
        return {int(value)}
    if isinstance(value, (list, tuple, set)):
        return {int(v) for v in value}
    text = str(value).strip()
    if not text or text.lower() in {"all", "none", "*"}:
        return None
    selected = set()
    for part in text.split(","):
        part = part.strip()
        if not part:
            continue
        if "-" in part:
            a, b = part.split("-", 1)
            a, b = int(a.strip()), int(b.strip())
            lo, hi = min(a, b), max(a, b)
            selected.update(range(lo, hi + 1))
        else:
            selected.add(int(part))
    return selected

def _parse_eval_rounds(value):
    return _parse_eval_tasks(value)

def _select_eval_clients(client_ids, fraction, mode="deterministic", seed=42):
    client_ids = sorted(int(cid) for cid in client_ids)
    if not client_ids:
        return client_ids
    fraction = float(fraction)
    if fraction <= 0:
        raise ValueError(f"EVAL_CLIENT_FRACTION must be > 0, got {fraction}")
    if fraction >= 1.0:
        return client_ids
    keep = max(1, int(round(len(client_ids) * fraction)))
    mode = str(mode).lower()
    if mode == "random":
        rng = np.random.default_rng(int(seed))
        return sorted(int(cid) for cid in rng.choice(client_ids, size=keep, replace=False).tolist())
    step = len(client_ids) / keep
    idxs = sorted(set(min(len(client_ids) - 1, int(round(k * step))) for k in range(keep)))
    while len(idxs) < keep:
        for idx in range(len(client_ids)):
            if idx not in idxs:
                idxs.append(idx)
                break
    return [client_ids[i] for i in sorted(idxs[:keep])]

def list_round_checkpoints(root):
    if not os.path.exists(root):
        raise FileNotFoundError(f"CHECKPOINT_ROOT does not exist: {root}")

    pattern = re.compile(r"checkpoint_task_(\d+)_round_(\d+)\.pt$")
    checkpoints = []
    for dirpath, _, files in os.walk(root):
        for filename in files:
            m = pattern.match(filename)
            if not m:
                continue
            path = os.path.join(dirpath, filename)
            checkpoints.append(
                {
                    "task_id": int(m.group(1)),
                    "round_id": int(m.group(2)),
                    "path": path,
                    "mtime": os.path.getmtime(path),
                }
            )
    checkpoints = sorted(checkpoints, key=lambda r: (r["task_id"], r["round_id"], r["path"]))
    if not checkpoints:
        raise FileNotFoundError(f"No checkpoint_task_<task>_round_<round>.pt found under {root}")
    return checkpoints

def _build_torch_context_router(context_detector, device):
    if context_detector is None or not getattr(context_detector, "context_learners", None):
        return None
    router = []
    for k, clf in enumerate(context_detector.context_learners):
        if clf is None or not hasattr(clf, "coef_") or not hasattr(clf, "intercept_"):
            router.append(None)
            continue
        coef = torch.as_tensor(clf.coef_[0], dtype=torch.float32, device=device)
        intercept = torch.as_tensor(float(clf.intercept_[0]), dtype=torch.float32, device=device)
        mask_np = context_detector.context_masks.get(k)
        if mask_np is None or not np.asarray(mask_np).any():
            mask_np = np.ones(int(coef.numel()), dtype=bool)
        mask_np = np.asarray(mask_np).astype(bool)
        if int(mask_np.sum()) != int(coef.numel()):
            # Fallback for old/incomplete checkpoints where mask dimensions drift.
            mask_np = np.ones(int(coef.numel()), dtype=bool)
        mask = torch.as_tensor(mask_np, dtype=torch.bool, device=device)
        router.append({"coef": coef, "intercept": intercept, "mask": mask})
    return router


def _binarize_context_activations_torch(context_detector, context_activations, device):
    parts = []
    thresholds = getattr(context_detector, "binarize_thresholds", None)
    for name in ["conv1", "conv2", "conv3", "gru"]:
        act = context_activations[name].detach().to(device)
        if thresholds is not None and name in thresholds:
            threshold = torch.as_tensor(float(thresholds[name]), dtype=act.dtype, device=device)
            binary = (act > threshold).float()
        else:
            binary = (act > 0).float()
        parts.append(binary)
    return torch.cat(parts, dim=1)


def _predict_episodes_torch(context_detector, router, binary_acts):
    latest_episode = max(context_detector.episode_classes.keys()) if context_detector.episode_classes else 0
    if not router:
        return torch.full((binary_acts.shape[0],), latest_episode, dtype=torch.long, device=binary_acts.device)

    pos_probs = []
    for entry in router:
        if entry is None:
            pos_probs.append(torch.zeros(binary_acts.shape[0], dtype=torch.float32, device=binary_acts.device))
            continue
        mask = entry["mask"]
        if mask.numel() == binary_acts.shape[1]:
            x = binary_acts[:, mask]
        else:
            x = binary_acts[:, : entry["coef"].numel()]
        logits = x.matmul(entry["coef"]) + entry["intercept"]
        pos_probs.append(torch.sigmoid(logits))

    pos = torch.stack(pos_probs, dim=1)
    neg = 1.0 - pos
    chain = torch.zeros(
        (binary_acts.shape[0], len(router) + 1),
        dtype=torch.float32,
        device=binary_acts.device,
    )
    for episode_index in range(len(router)):
        if episode_index == 0:
            chain[:, 0] = pos[:, 0]
        else:
            chain[:, episode_index] = torch.prod(neg[:, :episode_index], dim=1) * pos[:, episode_index]
    chain[:, -1] = torch.clamp(1.0 - chain.sum(dim=1), min=0.0)
    return torch.argmax(chain, dim=1)


def _apply_nice_context_mask_gpu(model, logits, context_detector, seen_classes, device, context_activations, router):
    if router is None or context_activations is None:
        return None
    if not getattr(context_detector, "episode_classes", None):
        return None

    binary_acts = _binarize_context_activations_torch(context_detector, context_activations, device)
    pred_episodes = _predict_episodes_torch(context_detector, router, binary_acts)
    masked = logits.clone()
    num_classes = masked.shape[1]
    seen_set = {int(c) for c in seen_classes}

    for episode in torch.unique(pred_episodes).tolist():
        allowed = [
            int(c)
            for c in context_detector.episode_classes.get(int(episode), [])
            if int(c) in seen_set and 0 <= int(c) < num_classes
        ]
        if not allowed:
            allowed = sorted(c for c in seen_set if 0 <= c < num_classes)
        if allowed:
            rows = pred_episodes == int(episode)
            row_idx = torch.nonzero(rows, as_tuple=False).flatten()
            if row_idx.numel() > 0:
                class_idx = torch.as_tensor(allowed, dtype=torch.long, device=device)
                masked[row_idx[:, None], class_idx] += 99999.0
    return masked

def _dict_get_int(mapping, key, default=None):
    if mapping is None:
        return default
    if key in mapping:
        return mapping[key]
    text_key = str(key)
    if text_key in mapping:
        return mapping[text_key]
    return default


def _make_denice_context_detector(ckpt, client_id):
    config = dict(ckpt["config"])
    algorithm_states = ckpt.get("client_algorithm_states", {})
    client_alg = _dict_get_int(algorithm_states, int(client_id), {}) or {}
    denice_state = client_alg.get("denice", client_alg)
    context_detector = ContextDetector(
        memo_per_class=int(config.get("memo_per_class", 50))
    )
    restore_context_detector(context_detector, denice_state.get("context_detector"))
    return context_detector


def _build_shared_context_detector_eval(
    context_detectors,
    memo_per_class,
    max_per_episode,
    seed,
    client_ids=None,
):
    selected = (
        list(context_detectors.values())
        if client_ids is None
        else [context_detectors[cid] for cid in client_ids if cid in context_detectors]
    )
    detectors = [det for det in selected if getattr(det, "activation_memory", None)]
    if not detectors:
        return None
    shared = build_pooled_context_detector(
        detectors,
        memo_per_class=memo_per_class,
        max_per_episode=max_per_episode,
        seed=seed,
    )
    if not getattr(shared, "activation_memory", None):
        return None
    return shared


def _normalize_context_groups(groups):
    normalized = {}
    for key, value in (groups or {}).items():
        try:
            cid = int(key)
        except Exception:
            continue
        normalized[cid] = sorted(set(int(v) for v in (value or [])))
    return normalized


def _float_stats(values):
    vals = [float(v) for v in values if v is not None]
    if not vals:
        return {"min": None, "mean": None, "max": None}
    return {"min": min(vals), "mean": sum(vals) / len(vals), "max": max(vals)}



def _make_equal_client_test_splits(test_X, test_y, client_ids, seed):
    """Split cumulative/global test evenly across current task clients for proxy eval."""
    client_ids = sorted(int(cid) for cid in client_ids)
    if not client_ids:
        return {}
    rng = np.random.default_rng(int(seed))
    indices = np.arange(len(test_y))
    rng.shuffle(indices)
    chunks = np.array_split(indices, len(client_ids))
    splits = {}
    for cid, chunk in zip(client_ids, chunks):
        idx = torch.as_tensor(chunk, dtype=torch.long)
        splits[int(cid)] = {
            "X_test": test_X.index_select(0, idx),
            "y_test": test_y.index_select(0, idx),
        }
    return splits


def _label_to_episode_map_eval(context_detector):
    mapping = {}
    for episode, classes in getattr(context_detector, "episode_classes", {}).items():
        for cls_id in classes:
            mapping[int(cls_id)] = int(episode)
    return mapping


def evaluate_denice_model_with_preds(
    model,
    test_data,
    device,
    context_detector,
    seen_classes,
    batch_size=1024,
    progress_label=None,
    progress_every_batches=0,
):
    """Same DeNICE eval path, but also return y_true/y_pred for confusion matrix."""
    model.eval()
    X_test = test_data["X_test"]
    y_test = test_data["y_test"]
    if len(y_test) == 0:
        empty = np.asarray([], dtype=np.int64)
        return {
            "loss": 0.0,
            "accuracy": 0.0,
            "precision_macro": 0.0,
            "recall_macro": 0.0,
            "f1_macro": 0.0,
            "f1_weighted": 0.0,
            "route_accuracy": 0.0,
            "route_coverage": 0.0,
        }, empty, empty

    criterion = nn.CrossEntropyLoss()
    all_preds = []
    all_targets = []
    total_loss = 0.0
    n_batches = int(np.ceil(len(y_test) / max(1, batch_size)))
    start_time = time.time()
    label2episode = _label_to_episode_map_eval(context_detector)
    route_correct = 0
    route_total = 0

    with torch.no_grad():
        for batch_idx, i in enumerate(range(0, len(y_test), batch_size), start=1):
            X_batch = X_test[i : i + batch_size].to(device)
            y_batch = y_test[i : i + batch_size].to(device)
            routed_logits, episodes = _denice_routed_logits_with_episodes(
                model, X_batch, context_detector, seen_classes, device
            )
            total_loss += criterion(routed_logits, y_batch).item() * len(y_batch)
            preds = routed_logits.argmax(dim=1)
            all_preds.extend(preds.detach().cpu().tolist())
            all_targets.extend(y_batch.detach().cpu().tolist())

            if episodes is not None and label2episode:
                y_np = y_batch.detach().cpu().numpy()
                true_eps = np.array([label2episode.get(int(c), -1) for c in y_np], dtype=np.int64)
                known = true_eps >= 0
                route_total += int(known.sum())
                route_correct += int((episodes[known] == true_eps[known]).sum())

            if progress_label and progress_every_batches > 0:
                should_print = batch_idx == 1 or batch_idx == n_batches or batch_idx % progress_every_batches == 0
                if should_print:
                    elapsed = time.time() - start_time
                    print(
                        f"      {progress_label}: batch {batch_idx}/{n_batches} "
                        f"({min(i + batch_size, len(y_test))}/{len(y_test)} samples), elapsed={elapsed:.1f}s",
                        flush=True,
                    )

    y_true = np.asarray(all_targets)
    y_pred = np.asarray(all_preds)
    metrics = {
        "loss": total_loss / max(1, len(y_test)),
        "accuracy": accuracy_score(y_true, y_pred),
        "precision_macro": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "recall_macro": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "f1_weighted": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "route_accuracy": (route_correct / route_total) if route_total > 0 else 0.0,
        "route_coverage": (route_total / len(y_test)) if len(y_test) > 0 else 0.0,
    }
    return metrics, y_true, y_pred


def evaluate_one_checkpoint(checkpoint_path, data_loader, test_cache, device, eval_batch_size, split_seed=None):
    # Use load_denice_checkpoint to handle both delta and non-delta checkpoints
    ckpt = load_denice_checkpoint(checkpoint_path)
    config = dict(ckpt["config"])
    config["data_dir"] = DATA_DIR
    ckpt["config"] = config

    task_id = int(ckpt.get("task_id", config.get("task_end", 0)))
    round_id = int(ckpt.get("round_id", ckpt.get("final_round_id", -1)))
    algorithm = str(ckpt.get("algorithm", config.get("algorithm", ""))).lower()
    num_classes = int(config.get("total_classes", config.get("num_classes", 34)))

    if task_id not in test_cache:
        test_cache[task_id] = data_loader.get_test_data(task_id, cumulative=True)
    test_X, test_y = test_cache[task_id]
    if len(test_y) == 0:
        raise ValueError(
            f"No test samples for task {task_id}. Check DATA_DIR={DATA_DIR} and global_test_data.npz"
        )

    seen_classes = ckpt.get("seen_classes") or list(range(num_classes))
    labels = list(range(num_classes))
    stored_metrics = ckpt.get("metrics", {}) or {}

    # =================================================================
    # DeNICE (decentralized) path: per-client models
    # =================================================================
    is_denice = (
        algorithm == "denice"
        and "client_model_states" in ckpt
        and "model_state_dict" not in ckpt
    )
    if is_denice:
        client_ids = [
            int(cid)
            for cid in ckpt.get("client_ids", ckpt["client_model_states"].keys())
        ]
        original_client_count = len(client_ids)
        client_ids = _select_eval_clients(
            client_ids,
            EVAL_CLIENT_FRACTION,
            mode=EVAL_CLIENT_SELECTION,
            seed=EVAL_CLIENT_SEED + task_id * 1000 + round_id,
        )
        print(
            f"Client filter: {len(client_ids)}/{original_client_count} clients "
            f"(fraction={EVAL_CLIENT_FRACTION}, selection={EVAL_CLIENT_SELECTION})",
            flush=True,
        )
        context_detectors = {
            int(cid): _make_denice_context_detector(ckpt, int(cid))
            for cid in client_ids
        }
        use_shared_context = bool(config.get("denice_shared_context_eval", True))
        shared_context_scope = str(config.get("denice_shared_context_scope", "cluster")).lower()
        shared_context_memo_per_class = int(config.get("nice_memo_per_class", config.get("memo_per_class", 50)))
        shared_max_raw = config.get("denice_shared_context_max_per_episode")
        shared_context_max_per_episode = None if shared_max_raw is None else int(shared_max_raw)
        shared_context_seed = int(config.get("random_seed", config.get("seed", 42)))
        context_groups = _normalize_context_groups((ckpt.get("cluster") or {}).get("groups", {}))

        shared_detector = None
        detector_cache = {}
        if use_shared_context and shared_context_scope == "global":
            shared_detector = _build_shared_context_detector_eval(
                context_detectors,
                memo_per_class=shared_context_memo_per_class,
                max_per_episode=shared_context_max_per_episode,
                seed=shared_context_seed,
            )

        if not use_shared_context or shared_context_scope == "local":
            routing_mode = "per-client"
        elif shared_context_scope == "global" and shared_detector is not None:
            routing_mode = "global"
        elif shared_context_scope == "cluster":
            routing_mode = "cluster"
        else:
            routing_mode = "per-client"

        def select_detector(cid, fallback_detector):
            detector = context_detectors.get(int(cid), fallback_detector)
            if routing_mode == "global" and shared_detector is not None:
                return shared_detector
            if routing_mode == "cluster":
                group_ids = context_groups.get(int(cid), [int(cid)])
                if int(cid) not in group_ids:
                    group_ids = [int(cid), *group_ids]
                group_key = tuple(sorted(set(int(x) for x in group_ids)))
                if group_key not in detector_cache:
                    detector_cache[group_key] = _build_shared_context_detector_eval(
                        context_detectors,
                        memo_per_class=shared_context_memo_per_class,
                        max_per_episode=shared_context_max_per_episode,
                        seed=shared_context_seed + int(cid),
                        client_ids=list(group_key),
                    )
                return detector_cache[group_key] or detector
            return detector

        print(
            f"Eval checkpoint start: task={task_id} round={round_id}, "
            f"clients={len(client_ids)}, test_samples={len(test_y)}, split_seed={split_seed}, "
            f"batch_size={eval_batch_size}, routing={routing_mode}, split=equal-client",
            flush=True,
        )
        checkpoint_eval_start = time.time()
        client_test_splits = _make_equal_client_test_splits(
            test_X, test_y, client_ids, EVAL_CLIENT_SEED if split_seed is None else split_seed
        )
        per_client_metrics = []
        per_client = {}
        checkpoint_y_true = []
        checkpoint_y_pred = []
        for pos, cid in enumerate(client_ids, start=1):
            client_eval_start = time.time()
            model, local_detector = _make_denice_client_model(ckpt, cid, device)
            detector = select_detector(cid, local_detector)
            client_data = client_test_splits[int(cid)]
            client_m, client_y_true, client_y_pred = evaluate_denice_model_with_preds(
                model,
                client_data,
                device,
                context_detector=detector,
                seen_classes=seen_classes,
                batch_size=eval_batch_size,
                progress_label=f"task={task_id} round={round_id} seed={split_seed} client={pos}/{len(client_ids)} cid={cid}",
                progress_every_batches=EVAL_PROGRESS_EVERY_BATCHES,
            )
            client_m = {k: float(v) for k, v in client_m.items()}
            client_m["eval_samples"] = int(len(client_data["y_test"]))
            checkpoint_y_true.append(client_y_true)
            checkpoint_y_pred.append(client_y_pred)
            per_client_metrics.append(client_m)
            per_client[int(cid)] = client_m
            if (
                EVAL_PROGRESS_EVERY_CLIENTS > 0
                and (pos == 1 or pos == len(client_ids) or pos % EVAL_PROGRESS_EVERY_CLIENTS == 0)
            ):
                elapsed = time.time() - checkpoint_eval_start
                client_elapsed = time.time() - client_eval_start
                pct = 100.0 * pos / max(1, len(client_ids))
                print(
                    f"  Eval client {pos}/{len(client_ids)} cid={cid} done "
                    f"({pct:.1f}%): acc={client_m['accuracy'] * 100:.2f}%, "
                    f"f1w={client_m['f1_weighted'] * 100:.2f}%, "
                    f"route_acc={client_m.get('route_accuracy', 0.0) * 100:.2f}%, "
                    f"time={client_elapsed:.1f}s, elapsed={elapsed:.1f}s",
                    flush=True,
                )
            del model, local_detector
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        metric_keys = [
            "loss",
            "accuracy",
            "precision_macro",
            "recall_macro",
            "f1_macro",
            "f1_weighted",
            "route_accuracy",
            "route_coverage",
        ]
        mean_m = {
            k: sum(float(row.get(k, 0.0)) for row in per_client_metrics) / max(1, len(per_client_metrics))
            for k in metric_keys
        }
        metrics = OrderedDict(
            checkpoint=checkpoint_path,
            algorithm=algorithm,
            task_id=task_id,
            round_id=round_id,
            train_loss=stored_metrics.get("train_loss"),
            test_loss=mean_m["loss"],
            accuracy=mean_m["accuracy"],
            precision_macro=mean_m["precision_macro"],
            recall_macro=mean_m["recall_macro"],
            f1_macro=mean_m["f1_macro"],
            f1_weighted=mean_m["f1_weighted"],
            route_accuracy=mean_m["route_accuracy"],
            route_coverage=mean_m["route_coverage"],
            routing_mode=routing_mode,
            eval_client_count=len(per_client_metrics),
            eval_client_total=original_client_count,
            eval_client_fraction=EVAL_CLIENT_FRACTION,
            eval_client_selection=EVAL_CLIENT_SELECTION,
            split_seed=split_seed,
            split_mode="equal_client",
            split_count=len(client_ids),
            cluster_K=(ckpt.get("cluster") or {}).get("K_t"),
            per_client_accuracy_stats=_float_stats([m.get("accuracy") for m in per_client_metrics]),
            per_client_route_accuracy_stats=_float_stats([m.get("route_accuracy") for m in per_client_metrics]),
            per_client=per_client,
        )

        y_true = np.concatenate(checkpoint_y_true) if checkpoint_y_true else np.asarray([], dtype=np.int64)
        y_pred = np.concatenate(checkpoint_y_pred) if checkpoint_y_pred else np.asarray([], dtype=np.int64)
        return metrics, y_true, y_pred, labels

    # =================================================================
    # Centralized path (NICE / DER / baseline)
    # =================================================================
    model, context_detector = _make_model(ckpt, device)
    torch_router = _build_torch_context_router(context_detector, device)
    criterion = nn.CrossEntropyLoss(reduction="sum")

    all_preds = []
    all_targets = []
    total_loss = 0.0
    model.eval()
    with torch.no_grad():
        for start in range(0, len(test_y), eval_batch_size):
            X_batch = test_X[start:start + eval_batch_size].to(device, non_blocking=True)
            y_batch = test_y[start:start + eval_batch_size].to(device, non_blocking=True)
            if (
                context_detector is not None
                and seen_classes is not None
                and hasattr(model, "get_output_and_context_activations")
            ):
                logits, context_activations = model.get_output_and_context_activations(X_batch)
            else:
                logits = model(X_batch)
                context_activations = None

            loss_logits = logits.clone()
            if context_detector is not None and seen_classes is not None:
                global_unseen = _nice_seen_mask(model, seen_classes, device)
                if len(global_unseen) == loss_logits.shape[1]:
                    loss_logits[:, global_unseen] = float("-inf")
                pred_logits = None
                try:
                    pred_logits = _apply_nice_context_mask_gpu(
                        model,
                        logits,
                        context_detector,
                        seen_classes,
                        device,
                        context_activations,
                        torch_router,
                    )
                except Exception as exc:
                    pred_logits = None
                if pred_logits is None:
                    pred_logits = _apply_local_nice_context_mask(
                        model,
                        logits,
                        X_batch,
                        context_detector,
                        seen_classes,
                        device,
                        context_activations=context_activations,
                    )
            else:
                pred_logits = logits

            total_loss += criterion(loss_logits, y_batch).item() * int(y_batch.shape[0])
            all_preds.append(pred_logits.argmax(dim=1).detach().cpu().numpy())
            all_targets.append(y_batch.detach().cpu().numpy())
            del X_batch, y_batch, logits, pred_logits, loss_logits

    y_true = np.concatenate(all_targets)
    y_pred = np.concatenate(all_preds)
    metrics = OrderedDict(
        checkpoint=checkpoint_path,
        algorithm=algorithm,
        task_id=task_id,
        round_id=round_id,
        train_loss=stored_metrics.get("train_loss"),
        test_loss=total_loss / max(1, len(y_true)),
        accuracy=accuracy_score(y_true, y_pred),
        precision_macro=precision_score(y_true, y_pred, average="macro", labels=labels, zero_division=0),
        recall_macro=recall_score(y_true, y_pred, average="macro", labels=labels, zero_division=0),
        f1_macro=f1_score(y_true, y_pred, average="macro", labels=labels, zero_division=0),
        f1_weighted=f1_score(y_true, y_pred, average="weighted", labels=labels, zero_division=0),
    )
    return metrics, y_true, y_pred, labels

DATA_DIR = resolve_data_dir(DATA_DIR)
checkpoints = list_round_checkpoints(CHECKPOINT_ROOT)
selected_tasks = _parse_eval_tasks(EVAL_TASKS)
if selected_tasks is not None:
    before = len(checkpoints)
    checkpoints = [c for c in checkpoints if int(c["task_id"]) in selected_tasks]
    if not checkpoints:
        raise FileNotFoundError(f"No checkpoints found for EVAL_TASKS={sorted(selected_tasks)} under {CHECKPOINT_ROOT}")
    print(f"Task filter: {sorted(selected_tasks)} -> {len(checkpoints)}/{before} checkpoints")
else:
    print("Task filter: all tasks")
selected_rounds = _parse_eval_rounds(EVAL_ROUNDS)
if selected_rounds is not None:
    before = len(checkpoints)
    checkpoints = [c for c in checkpoints if int(c["round_id"]) in selected_rounds]
    if not checkpoints:
        raise FileNotFoundError(f"No checkpoints found for EVAL_ROUNDS={sorted(selected_rounds)} under {CHECKPOINT_ROOT}")
    print(f"Round filter: {sorted(selected_rounds)} -> {len(checkpoints)}/{before} checkpoints")
else:
    print("Round filter: all rounds")
print(f"Client fraction: {EVAL_CLIENT_FRACTION} ({EVAL_CLIENT_SELECTION})")
print(f"Found {len(checkpoints)} round checkpoints")
print("First:", checkpoints[0]["path"])
print("Last :", checkpoints[-1]["path"])

device = "cuda" if torch.cuda.is_available() else "cpu"
eval_batch_size = int(globals().get("eval_batch_size", globals().get("CONFIG", {}).get("eval_batch_size", 32768)))
print("device:", device)
print("eval_batch_size:", eval_batch_size)

data_loader = IncrementalDataLoader(data_dir=DATA_DIR)
test_cache = {}
all_seed_rows = []
all_client_seed_rows = []
final_payload = None

METRIC_COLS = [
    "test_loss",
    "accuracy",
    "precision_macro",
    "recall_macro",
    "f1_macro",
    "f1_weighted",
    "route_accuracy",
    "route_coverage",
]


def _aggregate_seed_rows(seed_rows):
    if not seed_rows:
        return []
    df = pd.DataFrame(seed_rows)
    group_cols = ["checkpoint", "algorithm", "task_id", "round_id"]
    rows = []
    for key, g in df.groupby(group_cols, sort=True):
        row = OrderedDict(zip(group_cols, key))
        first = g.iloc[0].to_dict()
        row["train_loss"] = first.get("train_loss")
        row["routing_mode"] = first.get("routing_mode")
        row["eval_client_count"] = first.get("eval_client_count")
        row["eval_client_total"] = first.get("eval_client_total")
        row["eval_client_fraction"] = first.get("eval_client_fraction")
        row["eval_client_selection"] = first.get("eval_client_selection")
        row["split_mode"] = first.get("split_mode")
        row["split_count"] = first.get("split_count")
        row["split_seeds"] = sorted(int(v) for v in g["split_seed"].dropna().unique().tolist())
        row["num_seeds"] = int(len(row["split_seeds"]))
        row["cluster_K"] = first.get("cluster_K")
        for col in METRIC_COLS:
            vals = pd.to_numeric(g[col], errors="coerce") if col in g else pd.Series(dtype=float)
            mean = float(vals.mean()) if len(vals) else None
            std = float(vals.std(ddof=1)) if len(vals) > 1 else 0.0
            row[col] = mean
            row[f"{col}_mean"] = mean
            row[f"{col}_std"] = std
        rows.append(row)
    rows.sort(key=lambda r: (int(r["task_id"]), int(r["round_id"]), str(r["checkpoint"])))
    return rows


def _aggregate_client_seed_rows(client_seed_rows):
    if not client_seed_rows:
        return []
    df = pd.DataFrame(client_seed_rows)
    group_cols = ["checkpoint", "algorithm", "task_id", "round_id", "routing_mode", "client_id"]
    rows = []
    for key, g in df.groupby(group_cols, sort=True):
        row = OrderedDict(zip(group_cols, key))
        first = g.iloc[0].to_dict()
        row["eval_client_fraction"] = first.get("eval_client_fraction")
        row["eval_client_selection"] = first.get("eval_client_selection")
        row["split_seeds"] = sorted(int(v) for v in g["split_seed"].dropna().unique().tolist())
        row["num_seeds"] = int(len(row["split_seeds"]))
        metric_cols = ["loss", "accuracy", "precision_macro", "recall_macro", "f1_macro", "f1_weighted", "route_accuracy", "route_coverage", "eval_samples"]
        for col in metric_cols:
            if col not in g:
                continue
            vals = pd.to_numeric(g[col], errors="coerce")
            mean = float(vals.mean()) if len(vals) else None
            std = float(vals.std(ddof=1)) if len(vals) > 1 else 0.0
            row[col] = mean
            row[f"{col}_mean"] = mean
            row[f"{col}_std"] = std
        rows.append(row)
    rows.sort(key=lambda r: (int(r["task_id"]), int(r["round_id"]), int(r["client_id"])))
    return rows


def _autosave_partial_outputs(all_seed_rows, all_client_seed_rows, final_payload, status=None):
    """Persist progress after every checkpoint so Kaggle timeout does not lose work."""
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    all_rows = _aggregate_seed_rows(all_seed_rows)
    all_client_rows = _aggregate_client_seed_rows(all_client_seed_rows)

    all_json = os.path.join(OUTPUT_DIR, "all_round_metrics.json")
    all_csv = os.path.join(OUTPUT_DIR, "all_round_metrics.csv")
    with open(all_json, "w") as f:
        json.dump(all_rows, f, indent=2, default=str)
    pd.DataFrame(all_rows).to_csv(all_csv, index=False)

    seed_json = os.path.join(OUTPUT_DIR, "all_round_seed_metrics.json")
    seed_csv = os.path.join(OUTPUT_DIR, "all_round_seed_metrics.csv")
    with open(seed_json, "w") as f:
        json.dump(all_seed_rows, f, indent=2, default=str)
    pd.DataFrame(all_seed_rows).to_csv(seed_csv, index=False)

    client_json = os.path.join(OUTPUT_DIR, "all_round_client_metrics.json")
    client_csv = os.path.join(OUTPUT_DIR, "all_round_client_metrics.csv")
    with open(client_json, "w") as f:
        json.dump(all_client_rows, f, indent=2, default=str)
    pd.DataFrame(all_client_rows).to_csv(client_csv, index=False)

    client_seed_json = os.path.join(OUTPUT_DIR, "all_round_client_seed_metrics.json")
    client_seed_csv = os.path.join(OUTPUT_DIR, "all_round_client_seed_metrics.csv")
    with open(client_seed_json, "w") as f:
        json.dump(all_client_seed_rows, f, indent=2, default=str)
    pd.DataFrame(all_client_seed_rows).to_csv(client_seed_csv, index=False)

    if final_payload is not None:
        seed_final_metrics, _final_y_true, _final_y_pred, _final_labels = final_payload
        final_metrics = all_rows[-1] if all_rows else seed_final_metrics
        final_json = os.path.join(OUTPUT_DIR, "final_round_metrics.json")
        final_csv = os.path.join(OUTPUT_DIR, "final_round_metrics.csv")
        with open(final_json, "w") as f:
            json.dump(final_metrics, f, indent=2, default=str)
        pd.DataFrame([final_metrics]).to_csv(final_csv, index=False)

    status_payload = {
        "complete": False,
        "autosaved_seed_rows": len(all_seed_rows),
        "autosaved_client_seed_rows": len(all_client_seed_rows),
        "output_dir": OUTPUT_DIR,
    }
    if status:
        status_payload.update(status)
    status_json = os.path.join(OUTPUT_DIR, "autosave_status.json")
    with open(status_json, "w") as f:
        json.dump(status_payload, f, indent=2, default=str)
    return all_rows, all_client_rows


for seed_index, split_seed in enumerate(EVAL_SPLIT_SEEDS, start=1):
    print(f"=== Split seed {seed_index}/{len(EVAL_SPLIT_SEEDS)}: {split_seed} ===", flush=True)
    for idx, item in enumerate(checkpoints, start=1):
        print(
            f"Starting checkpoint {idx}/{len(checkpoints)} seed={split_seed}: "
            f"task={item['task_id']} round={item['round_id']} path={item['path']}",
            flush=True,
        )
        metrics, y_true, y_pred, labels = evaluate_one_checkpoint(
            item["path"], data_loader, test_cache, device, eval_batch_size, split_seed=split_seed
        )
        metrics["split_seed"] = int(split_seed)
        metrics["seed_index"] = int(seed_index)
        all_seed_rows.append(metrics)
        for cid, row in (metrics.get("per_client") or {}).items():
            all_client_seed_rows.append({
                "checkpoint": metrics["checkpoint"],
                "algorithm": metrics["algorithm"],
                "task_id": metrics["task_id"],
                "round_id": metrics["round_id"],
                "routing_mode": metrics.get("routing_mode"),
                "eval_client_fraction": metrics.get("eval_client_fraction"),
                "eval_client_selection": metrics.get("eval_client_selection"),
                "split_seed": int(split_seed),
                "seed_index": int(seed_index),
                "client_id": int(cid),
                **row,
            })
        final_payload = (metrics, y_true, y_pred, labels)
        print(
            f"[{idx}/{len(checkpoints)} seed={split_seed}] "
            f"task={metrics['task_id']} round={metrics['round_id']} "
            f"acc={metrics['accuracy'] * 100:.2f}% "
            f"f1w={metrics['f1_weighted'] * 100:.2f}%"
        )
        _autosave_partial_outputs(
            all_seed_rows,
            all_client_seed_rows,
            final_payload,
            status={
                "last_seed": int(split_seed),
                "seed_index": int(seed_index),
                "num_seeds": int(len(EVAL_SPLIT_SEEDS)),
                "checkpoint_index": int(idx),
                "num_checkpoints": int(len(checkpoints)),
                "last_task_id": int(metrics["task_id"]),
                "last_round_id": int(metrics["round_id"]),
                "last_checkpoint": metrics["checkpoint"],
            },
        )
        print(f"  Autosaved partial outputs to {OUTPUT_DIR}", flush=True)
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

all_rows = _aggregate_seed_rows(all_seed_rows)
all_client_rows = _aggregate_client_seed_rows(all_client_seed_rows)

# Summary outputs keep the old filenames. Raw per-seed outputs are saved separately.
all_json = os.path.join(OUTPUT_DIR, "all_round_metrics.json")
all_csv = os.path.join(OUTPUT_DIR, "all_round_metrics.csv")
with open(all_json, "w") as f:
    json.dump(all_rows, f, indent=2, default=str)
pd.DataFrame(all_rows).to_csv(all_csv, index=False)

seed_json = os.path.join(OUTPUT_DIR, "all_round_seed_metrics.json")
seed_csv = os.path.join(OUTPUT_DIR, "all_round_seed_metrics.csv")
with open(seed_json, "w") as f:
    json.dump(all_seed_rows, f, indent=2, default=str)
pd.DataFrame(all_seed_rows).to_csv(seed_csv, index=False)

client_json = os.path.join(OUTPUT_DIR, "all_round_client_metrics.json")
client_csv = os.path.join(OUTPUT_DIR, "all_round_client_metrics.csv")
with open(client_json, "w") as f:
    json.dump(all_client_rows, f, indent=2, default=str)
pd.DataFrame(all_client_rows).to_csv(client_csv, index=False)

client_seed_json = os.path.join(OUTPUT_DIR, "all_round_client_seed_metrics.json")
client_seed_csv = os.path.join(OUTPUT_DIR, "all_round_client_seed_metrics.csv")
with open(client_seed_json, "w") as f:
    json.dump(all_client_seed_rows, f, indent=2, default=str)
pd.DataFrame(all_client_seed_rows).to_csv(client_seed_csv, index=False)
with open(os.path.join(OUTPUT_DIR, "autosave_status.json"), "w") as f:
    json.dump({
        "complete": True,
        "autosaved_seed_rows": len(all_seed_rows),
        "autosaved_client_seed_rows": len(all_client_seed_rows),
        "num_seeds": len(EVAL_SPLIT_SEEDS),
        "num_checkpoints": len(checkpoints),
        "output_dir": OUTPUT_DIR,
    }, f, indent=2, default=str)

# Backward-compatible final metrics files: summary of last checkpoint in sorted task/round order.
_seed_final_metrics, final_y_true, final_y_pred, final_labels = final_payload
final_metrics = all_rows[-1] if all_rows else _seed_final_metrics
final_json = os.path.join(OUTPUT_DIR, "final_round_metrics.json")
final_csv = os.path.join(OUTPUT_DIR, "final_round_metrics.csv")
with open(final_json, "w") as f:
    json.dump(final_metrics, f, indent=2, default=str)
pd.DataFrame([final_metrics]).to_csv(final_csv, index=False)

cm = confusion_matrix(final_y_true, final_y_pred, labels=final_labels)
cm_csv_path = os.path.join(OUTPUT_DIR, "confusion_matrix_34.csv")
cm_png_path = os.path.join(OUTPUT_DIR, "confusion_matrix_34.png")
cm_pdf_path = os.path.join(OUTPUT_DIR, "confusion_matrix_34.pdf")
pd.DataFrame(cm, index=final_labels, columns=final_labels).to_csv(cm_csv_path)

plt.figure(figsize=(24, 20))
sns.heatmap(cm, cmap="Blues", xticklabels=final_labels, yticklabels=final_labels, cbar=True)
plt.xlabel("Predicted class")
plt.ylabel("True class")
plt.title(
    f"Confusion Matrix - {len(final_labels)} classes - "
    f"task {final_metrics['task_id']} round {final_metrics['round_id']} "
    f"(last seed={_seed_final_metrics.get('split_seed')})"
)
plt.tight_layout()
plt.savefig(cm_png_path, dpi=200)
plt.savefig(cm_pdf_path)
plt.show()

print("Saved all-round summary metrics JSON:", all_json)
print("Saved all-round summary metrics CSV:", all_csv)
print("Saved all-round seed metrics JSON:", seed_json)
print("Saved all-round seed metrics CSV:", seed_csv)
print("Saved all-round client summary metrics JSON:", client_json)
print("Saved all-round client summary metrics CSV:", client_csv)
print("Saved all-round client seed metrics JSON:", client_seed_json)
print("Saved all-round client seed metrics CSV:", client_seed_csv)
print("Saved final metrics JSON:", final_json)
print("Saved final metrics CSV:", final_csv)
print("Saved final confusion CSV:", cm_csv_path)
print("Saved final confusion PNG:", cm_png_path)
print("Saved final confusion PDF:", cm_pdf_path)
pd.DataFrame(all_rows).tail()


In [ ]:
import json
import os

import pandas as pd

# Scan all result folders and merge every final_round_metrics.json into one table.
METRICS_ROOTS = [
    globals().get("CHECKPOINT_ROOT", "/kaggle/input/datasets/khoilv2005/il-der/results_il_der"),
    globals().get("OUTPUT_DIR", "/kaggle/working/results_il_der_eval"),
]
MERGED_OUTPUT_DIR = globals().get("OUTPUT_DIR", "/kaggle/working/results_il_der_eval")
os.makedirs(MERGED_OUTPUT_DIR, exist_ok=True)

rows = []
seen_paths = set()
for root_dir in METRICS_ROOTS:
    if not root_dir or not os.path.exists(root_dir):
        continue
    for root, _, files in os.walk(root_dir):
        if "final_round_metrics.json" not in files:
            continue
        path = os.path.join(root, "final_round_metrics.json")
        if path in seen_paths:
            continue
        seen_paths.add(path)
        try:
            with open(path, "r") as f:
                record = json.load(f)
            record["metrics_file"] = path
            record["run_dir"] = root
            rows.append(record)
        except Exception as exc:
            print("Failed to load:", path, exc)

rows = sorted(
    rows,
    key=lambda r: (
        str(r.get("algorithm", "")),
        int(r.get("task_id", -1) or -1),
        int(r.get("round_id", -1) or -1),
        str(r.get("checkpoint", "")),
    ),
)

merged_json = os.path.join(MERGED_OUTPUT_DIR, "all_final_round_metrics.json")
merged_csv = os.path.join(MERGED_OUTPUT_DIR, "all_final_round_metrics.csv")
with open(merged_json, "w") as f:
    json.dump(rows, f, indent=2, default=str)
pd.DataFrame(rows).to_csv(merged_csv, index=False)

print(f"Merged {len(rows)} final_round_metrics.json files")
print("Saved merged JSON:", merged_json)
print("Saved merged CSV:", merged_csv)
pd.DataFrame(rows).head()
